In [1]:
pip install xarray cfgrib pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import xarray as xr
import numpy as np
import flox
import gc 
import os

In [3]:
grib_files = [x for x in os.listdir('Dataset')if x[-5:] == '.grib']
grib_files

['temperature_data.grib',
 'data.grib',
 'precipitation_data.grib',
 'evap_total_cloud_2005-2024.grib',
 'pressure_and_evaporation_data.grib']

In [4]:
for x in grib_files:
    #new_name = f'{x.split('.')[0]}.nc'
    print(x.split('.')[0])

temperature_data
data
precipitation_data
evap_total_cloud_2005-2024
pressure_and_evaporation_data


In [5]:
for x in grib_files:
    downsampled_files = os.listdir('downsampled_datasets')
    new_name = 's_' + x.split('.')[0] + '.nc'
    if new_name in downsampled_files:
        print(new_name + ': already downsampled')
    elif new_name not in downsampled_files:
        print(new_name + ': downsampling')
    
        file_path = 'Dataset/' + x
        try:
            ds = xr.open_dataset(file_path, engine='cfgrib')
            #print(ds)
        except ValueError as e:
            print(f"Error opening the GRIB file with cfgrib: {e}")
    
        indices = ['time', 'latitude', 'longitude']
        data_vars = list(ds.keys())
        relevant_vars = indices + data_vars
    
        df = ds[relevant_vars].to_dataframe().reset_index()[relevant_vars]
        df['time'] = df['time'].dt.floor('D')
    
        gc.collect()
    
        downsampled_df = df.groupby(indices, sort=False)[data_vars].mean()
    
        smaller_ds = xr.Dataset.from_dataframe(downsampled_df)
        new_fp = f'/home/valau/DS3 Winter 2025 Project/downsampled_datasets/{new_name}'
        smaller_ds.to_netcdf(path=new_fp, mode='w')

        print(new_name + ': downsampled')
    elif x == grib_files[-1]:
        print('done')
        break

Can't read index file 'Dataset/pressure_and_evaporation_data.grib.5b7b6.idx'
Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", line 551, in from_indexpath_or_filestream
    self = cls.from_indexpath(indexpath)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", line 430, in from_indexpath
    index = pickle.load(file)
            ^^^^^^^^^^^^^^^^^
EOFError: Ran out of input


s_temperature_data.nc: already downsampled
s_data.nc: already downsampled
s_precipitation_data.nc: already downsampled
s_evap_total_cloud_2005-2024.nc: already downsampled
s_pressure_and_evaporation_data.nc: downsampling


skipping variable: paramId==182 shortName='e'
Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='time' value=Variable(dimensions=('time',), data=array([1104537600, 1104559200, 1104580800, ..., 1735624800, 1735646400,
       1735668000])) new_value=Variable(dimensions=('time',), data=array([1104516000, 1104559200, 1104602400, ..., 1735538400, 1735581600,
       1735624800]))
skipping variable: paramId==228251 shortName='pev'
Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/home/valau/.local/lib/python3.11

s_pressure_and_evaporation_data.nc: downsampled


In [ ]:
# Replace 'your_file.grib' with the actual path to your GRIB file
#file_path = 'Dataset/evap_total_cloud_2005-2024.grib'

#try:
#    ds = xr.open_dataset(file_path, engine='cfgrib')
#    #print(ds)
#except ValueError as e:
#    print(f"Error opening the GRIB file with cfgrib: {e}")

In [ ]:
#indices = ['time', 'latitude', 'longitude']
#data_vars = list(ds.keys())
#relevant_vars = indices + data_vars
#relevant_vars

In [ ]:
#ds.groupby(coordinates).mean(dim='x')

In [ ]:
#df = ds[relevant_vars].to_dataframe().reset_index()[relevant_vars]
#df

In [ ]:
#df['time'] = df['time'].dt.floor('D')#[121000000]
#df#.head()

In [ ]:
#df['time'][15768]

In [ ]:
#gc.collect()

In [ ]:
#df['time'][15687]

In [ ]:
#downsampled_df = df.groupby(indices, sort=False)[data_vars].mean()#.reset_index()
#downsampled_df

In [ ]:
#smaller_ds = xr.Dataset.from_dataframe(downsampled_df)

In [ ]:
#smaller_ds.to_netcdf(path='/home/valau/DS3 Winter 2025 Project/downsampled_datasets/s_evap_total_cloud_2005-2024.nc', mode='w')

In [ ]:
# Replace 'your_file.grib' with the actual path to your GRIB file
#test_file_path = 'downsampled_datasets/s_evap_total_cloud_2005-2024.nc'

#try:
#    test_ds = xr.open_dataset(test_file_path)
#    #print(ds)
#except ValueError as e:
#    print(f"Error opening the GRIB file with cfgrib: {e}")

In [ ]:
#test_df = test_ds.to_dataframe().reset_index()
#test_df

#indices = ['date', 'latitude', 'longitude']
df['time'] = pd.to_datetime({
       'year': df['time'].dt.year,
       'month': df['time'].dt.month,
       'day': df['time'].dt.month,
       'hour': 0
   })
#df.drop(columns=['time'], inplace=True)
df.head()#['hour'] = df

#indices = ['date', 'latitude', 'longitude']
df['time'] = df['time'].dt.date# + pd.Timedelta(hours=12)
#df.drop(columns=['time'], inplace=True)
df.head()#['hour'] = df

downsampled_df.set_index(['date','latitude', 'longitude'], inplace=True)
downsampled_df

#indices = ['date', 'latitude', 'longitude']
#df['date'] = df['time'].dt.date + pd.Timedelta(hours=12)
#df.drop(columns=['time'], inplace=True)
df['time'] = pd.to_datetime({
       'year': df['time'].dt.year,
       'month': df['time'].dt.month,
       'day': df['time'].dt.month,
       'hour': 0
   })
#df.head()#['hour'] = df

# Specify the file path
#filepath = 'path/to/your/file.csv'

# Export DataFrame to CSV
#df.to_csv(filepath, index=False)

#for x in ds.keys():
#    print(x)
#ds['tcc']

#df['time'][15475].date()#.head()

#df['time'][15598].hour

#df['time'][:20].dt.date[0]